<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-22</br>
</div>

</br>

# 학습 내용
>이번 장에서는 <strong>ResNet-50 비교(ResNet-50 Comparison)</strong>에 대해 학습합니다.</br></br>
>ResNet-50으로 추론 파이프라인을 구성하고 CLIP과의 분류 결과를 학습해봅시다.

<div style="text-align:center">

</div>

</br>

# ResNet-50 추론 (Inference)
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">ImageNet 사전학습된 ResNet-50</mark>으로 이미지 분류를 수행합니다.

> 신경망을 단순히 깊게 쌓으면 성능이 오히려 저하되는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">기울기 소실(Vanishing Gradient)</mark> 문제가 발생합니다.</br></br>
> 역전파 시 기울기가 층을 거치며 점점 작아져 앞쪽 층의 가중치가 거의 업데이트되지 않기 때문입니다.</br></br>
> ResNet은 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">잔차 연결(Skip Connection)</mark>을 도입하여 이 문제를 해결합니다.</br>
> 입력을 변환하지 않고 출력에 직접 더하는 단축 경로(shortcut)를 통해 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">기울기가 깊은 층까지 직접 전달</mark>되므로, 50층 이상의 매우 깊은 네트워크도 안정적으로 학습할 수 있습니다.</br>

<div style="text-align:center">

</div>

<div style="text-align:center">

</div>

<div style="text-align:center">

</div>

In [4]:
# TODO 1: 사전학습된 ResNet-50 모델을 로드하고 평가 모드로 설정해봅시다.
import torchvision.models as models
import torch

weights=models.ResNet50_Weights.IMAGENET1K_V1
model = models.resnet50(weights=weights)
model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [5]:
# TODO 2: 해당 가중치에 맞는 전처리를 설정해봅시다.
preprocess = weights.transforms()


class_prompts = {
    "cat": "a photo of a cute domestic cat sitting, high quality, detailed",
    "dog": "a photo of a golden retriever dog sitting, high quality, detailed",
    "car": "a photo of a red sports car on a road, high quality, detailed",
    "airplane": "a photo of a commercial airplane flying in the sky, high quality, detailed",
    "flower": "a photo of a beautiful sunflower in a garden, high quality, detailed",
}

print(f"클래스 수: {len(class_prompts)}")
for cls, prompt in class_prompts.items():
    print(f"  {cls}: {prompt[:50]}...")

클래스 수: 5
  cat: a photo of a cute domestic cat sitting, high quali...
  dog: a photo of a golden retriever dog sitting, high qu...
  car: a photo of a red sports car on a road, high qualit...
  airplane: a photo of a commercial airplane flying in the sky...
  flower: a photo of a beautiful sunflower in a garden, high...


</br>

## 추론 파이프라인

In [7]:
# TODO 3: 이미지를 로드하고 전처리하여 배치 차원을 추가해봅시다.
from PIL import Image

image = Image.open('dog.jpg')
input_tensor = preprocess(image).unsqueeze(0)
input_tensor.shape


torch.Size([1, 3, 224, 224])

💡배치 차원과 unsqueeze(0)
> ResNet 같은 CNN 모델은 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">4차원 텐서 (B, C, H, W)</mark>를 입력으로 기대합니다.</br></br>
> ```
> 전처리 후:    (3, 224, 224)     → 채널, 높이, 너비 (이미지 1장)
> unsqueeze(0): (1, 3, 224, 224)  → 배치, 채널, 높이, 너비
> ```
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">배치 차원(B)</mark>은 "몇 장의 이미지를 동시에 처리할지"를 나타내는 첫 번째 축입니다.</br>
> 학습 시에는 32, 64장씩 묶어 처리하지만, 추론 시 1장이라도 `(1, ...)` 형태로 맞춰야 합니다.</br>
> `unsqueeze(0)` = 맨 앞에 크기 1인 차원을 추가하는 함수입니다.

In [11]:
# TODO 4: 추론 모드에서 모델을 실행하고 확률을 구해봅시다.
with torch.no_grad():
    output = model(input_tensor)
    values = torch.softmax(output, dim=1)

output.shape
values.sum()


tensor(1.0000)

In [15]:
# TODO 5: Top-5 예측 결과를 출력해봅시다.
value, indices = values.topk(5)
categories = weights.meta['categories']
for i in range(5):
    idx = indices[0][i].item()
    v = value[0][i].item()
    print(f"{i+1}. {categories[idx]}: {v:.4f}")

golden retriever 0.9700194597244263
Labrador retriever 0.021223613992333412
kuvasz 0.002047913148999214
Tibetan mastiff 0.0011571162613108754
tennis ball 0.0006430643261410296


</br>

## ResNet-50 vs CLIP 비교

💡weights.transforms()
> `weights.transforms()`는 해당 모델 학습 시 사용된 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">정확한 전처리</mark>를 반환합니다.</br>
> 수동으로 Normalize 값을 입력하는 것보다 안전합니다.

💡어떤 모델을 선택할까?
> 고정된 카테고리 분류: <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">ResNet-50</mark> (빠르고 정확)</br>
> 유연한 분류/검색: <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">CLIP</mark> (새 카테고리 추가 가능, Zero-shot)